# Initialize Store & Generate Tile List

Run this notebook **once** before any tile processing.

1. Loads config from `config/config_v1.txt` (use `config_with_secrets_v1.txt` for local runs with real credentials)
2. Builds a GeoDataFrame of all 10°×10° tiles with a `land` boolean column, saves `tile_list.geojson` — committed to the repo and not recomputed in CI
3. Creates the empty Icechunk/Zarr v3 store on Azure Blob Storage

After this, trigger the **Process All Tiles** GitHub Actions workflow to fill the store.

In [ ]:
import sys
from pathlib import Path

import dask.array as da
import geodatasets
import geopandas as gpd
import icechunk
import numpy as np
import xarray as xr
from shapely.geometry import box

sys.path.insert(0, str(Path.cwd().parent))
from config import Config

# Use config_with_secrets_v1.txt for local runs; config_v1.txt reads credentials from env vars.
cfg = Config("config/config_v1.txt")

print("YEARS:", cfg.YEARS)
print("RESOLUTION:", cfg.RESOLUTION)
print("TILE_SIZE_DEG:", cfg.TILE_SIZE_DEG)
print("PIXELS_PER_TILE:", cfg.PIXELS_PER_TILE)
print("Global geobox:", cfg.global_geobox)

## 1. Build tile list and save as GeoJSON

Creates a GeoDataFrame of **all** 10°×10° tiles with a `land` boolean column,
intersected against Natural Earth land polygons loaded via `geodatasets`.
Saves `tile_list.geojson` — commit this file to the repo. CI and notebook 02
read it directly (filtering to `land=True`) without recomputing.

In [ ]:
# Build GeoDataFrame of all tiles, deriving each geometry directly from its
# GeoBox boundary so the tile list is provably consistent with the global geobox.
tiles = []
for row in range(cfg.TILE_ROWS):
    for col in range(cfg.TILE_COLS):
        tile_geobox = cfg.tile_geobox(row, col)
        bb = tile_geobox.boundingbox
        tiles.append({
            "row": row, "col": col,
            "geometry": box(bb.left, bb.bottom, bb.right, bb.top),
        })

tiles_gdf = gpd.GeoDataFrame(tiles, crs="EPSG:4326")

# Natural Earth land polygons via geodatasets
land_gdf = gpd.read_file(geodatasets.get_url("naturalearth land"))

# Mark tiles that intersect land
tiles_gdf["land"] = tiles_gdf.intersects(land_gdf.union_all())

tile_list_path = Path.cwd().parent / "tile_list.geojson"
tiles_gdf.to_file(tile_list_path, driver="GeoJSON")
print(f"{tiles_gdf['land'].sum()} land tiles out of {len(tiles_gdf)} total written to {tile_list_path}")
tiles_gdf.head()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

f, ax = plt.subplots(figsize=(12, 7), dpi=150)

land_gdf.plot(ax=ax, color="green", zorder=0)
tiles_gdf.plot(
    ax=ax,
    facecolor=tiles_gdf["land"].map({True: "green", False: "none"}),
    edgecolor="black", alpha=0.5, zorder=1,
)

ax.set_title("Tile grid system")
ax.set_xlabel("Tile column number")
ax.set_ylabel("Tile row number")

h_values = sorted(tiles_gdf["col"].unique())
v_values = sorted(tiles_gdf["row"].unique(), reverse=True)
h_coords = [tiles_gdf[tiles_gdf["col"] == h].geometry.centroid.x.mean() for h in h_values]
v_coords = [tiles_gdf[tiles_gdf["row"] == v].geometry.centroid.y.mean() for v in v_values]

ax.set_xticks(h_coords)
ax.set_xticklabels([f"col{h}" for h in h_values], rotation=90)
ax.set_yticks(v_coords)
ax.set_yticklabels([f"row{v}" for v in v_values])
ax.tick_params(axis="both", which="both", length=0)
ax.set_xlim(tiles_gdf.total_bounds[0], tiles_gdf.total_bounds[2])
ax.set_ylim(tiles_gdf.total_bounds[1], tiles_gdf.total_bounds[3])

legend_elements = [
    Patch(facecolor="green", alpha=0.5, edgecolor="black",
          label=f'Land tile ({tiles_gdf["land"].sum()})'),
    Patch(facecolor="grey", alpha=0.5, edgecolor="black",
          label=f'Ocean tile ({(~tiles_gdf["land"]).sum()})'),
]
ax.legend(handles=legend_elements, loc="lower left")
f.tight_layout()

## 2. Initialize the Icechunk store

Credentials are read from `config/config_v1.txt` (values set to `ENV` are resolved
from environment variables). For local runs, switch to `config_with_secrets_v1.txt`
in the first cell above.

This creates only metadata and coordinates — no data chunks until tile runners fill them.

In [ ]:
n_lat  = cfg.TILE_ROWS * cfg.PIXELS_PER_TILE
n_lon  = cfg.TILE_COLS * cfg.PIXELS_PER_TILE
shape  = (len(cfg.YEARS), n_lat, n_lon)
chunks = (1, cfg.PIXELS_PER_TILE, cfg.PIXELS_PER_TILE)

lats = np.arange(90, -90, -cfg.RESOLUTION) - cfg.RESOLUTION / 2
lons = np.arange(-180, 180, cfg.RESOLUTION) + cfg.RESOLUTION / 2

var_attrs = {
    "scale_factor": np.float32(0.02),
    "add_offset": np.float32(0.0),
    "_FillValue": cfg.FILL_VALUE,
    "valid_range": [7500, 65535],
    "units": "K",
    "grid_mapping": "spatial_ref",
}

ds = xr.Dataset(
    {
        "avg_daytime_lst": xr.DataArray(
            da.full(shape, np.uint16(cfg.FILL_VALUE), dtype=np.uint16, chunks=chunks),
            dims=["year", "latitude", "longitude"],
            attrs={**var_attrs, "long_name": "Annual mean daytime land surface temperature"},
        ),
        "max_daytime_lst": xr.DataArray(
            da.full(shape, np.uint16(cfg.FILL_VALUE), dtype=np.uint16, chunks=chunks),
            dims=["year", "latitude", "longitude"],
            attrs={**var_attrs, "long_name": "Annual maximum daytime land surface temperature"},
        ),
    },
    coords={"year": np.array(cfg.YEARS), "latitude": lats, "longitude": lons},
)
ds.attrs = {
    "title": "MODIS MOD11A2 Annual Daytime Land Surface Temperature",
    "source": "MODIS Terra MOD11A2 Version 6.1 via Microsoft Planetary Computer",
    "Conventions": "CF-1.8",
}
print(ds)

In [ ]:
storage = icechunk.azure_storage(
    account=cfg.AZURE_STORAGE_ACCOUNT,
    container=cfg.AZURE_CONTAINER,
    prefix=cfg.ICECHUNK_PREFIX,
    sas_token=cfg.AZURE_STORAGE_SAS_TOKEN,
)

repo = icechunk.Repository.create(storage)
session = repo.writable_session("main")

ds.to_zarr(
    session.store,
    mode="w",
    zarr_format=3,
    compute=False,
    write_empty_chunks=False,
    consolidated=False,
)

snapshot_id = session.commit("initialize store: empty template")
print(f"Store initialized. Snapshot ID: {snapshot_id}")